## 1 Import python libraries and download origin dataset

In [1]:
import os
import pandas as pd
import numpy as np

# Define file paths
RAW_DATA_PATH = "../datasets/raw/top-spotify-songs-2023.csv"
PROCESSED_DATA_PATH = "../datasets/processed/cleaned_spotify_2023.csv"

print("--- Downloading origin dataset ---")
# using encoding='latin-1' to avoid errors with special characters in song/artist names.
# latin-1 helps avoid errors with special characters in song/artist names.
df = pd.read_csv(RAW_DATA_PATH, encoding='latin-1')

# Display initial overview of the dataframe
print(f"Initial dataset size: {df.shape[0]} rows, {df.shape[1]} columns\n")
print(df.info())

--- Downloading origin dataset ---
Initial dataset size: 953 rows, 24 columns

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    

## 2 Fix formatting errors and force the 'streams' column style

In [2]:
print("\n--- Currently preprocessing the column 'streams' ---")

# Detect rows that are not numbers (contain text errors due to misaligned rows when scraping data)
# errors='coerce' will automatically convert error text rows to NaN (Not a Number)
df['streams'] = pd.to_numeric(df['streams'], errors='coerce')

# Count how many rows have incorrect data types in the 'streams' column
null_streams_count = df['streams'].isnull().sum()
print(f"Found {null_streams_count} rows with incorrect format in the 'streams' column.")

# Drop rows with NaN values in the 'streams' column as it's the most important target variable
df = df.dropna(subset=['streams'])
# Cast the column to a larger integer type (int64) for memory optimization and computation
df['streams'] = df['streams'].astype('int64')
# Log transformation: to handle skewed data for OLS regression
df['log_streams'] = np.log1p(df['streams'])
print("Successfully created 'log_streams' column for linear regression assumptions.")


--- Currently preprocessing the column 'streams' ---
Found 1 rows with incorrect format in the 'streams' column.
Successfully created 'log_streams' column for linear regression assumptions.


## 3 Cleaning up corrupt data in columns from other platforms

In [3]:
print("\n--- Currently preprocessing the column 'in_deezer_playlists' and 'in_shazam_charts' ---")

# In this dataset, columns like 'in_deezer_playlists' or 'in_shazam_charts'
# often contain commas as thousands separators (e.g., "1,234") which causes Pandas to misinterpret them as strings.
columns_to_clean = [
    'in_spotify_playlists', 'in_apple_playlists', 'in_apple_charts',
    'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts'
]

for col in columns_to_clean:
    if col in df.columns:
        # Remove comma thousand separators if present
        df[col] = df[col].astype(str).str.replace(',', '')
        # Convert to numeric, coerce errors to NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Fill NaN values with 0 (assuming not in the playlist/charts)
        df[col] = df[col].fillna(0).astype('int64')


--- Currently preprocessing the column 'in_deezer_playlists' and 'in_shazam_charts' ---


## 4 Text Data Processing and Duplicate Handling

In [4]:
# Remove leading and trailing whitespace from 'track_name' and 'artist(s)_name' columns to ensure consistency
df['track_name'] = df['track_name'].str.strip()
df['artist(s)_name'] = df['artist(s)_name'].str.strip()

## 5 Handling missing values for the ‘key’ classification column

In [5]:
if 'key' in df.columns:
    df['key'] = df['key'].fillna('Unknown')

## 6 Checking and handling missing values in the main analysis columns

In [6]:
print("\n--- Checking Audio Features ---")

# Audio feature columns like 'danceability_%', 'energy_%' must be complete
audio_features = ['danceability_%', 'energy_%', 'valence_%', 'acousticness_%']

# Check for null values in these columns
for feature in audio_features:
    missing = df[feature].isnull().sum()
    if missing > 0:
        print(f"Column {feature} is missing {missing} rows. Proceeding to drop them.")
        df = df.dropna(subset=[feature])

print("Status of null values after processing:")
print(df[audio_features + ['streams']].isnull().sum())


--- Checking Audio Features ---
Status of null values after processing:
danceability_%    0
energy_%          0
valence_%         0
acousticness_%    0
streams           0
dtype: int64


## 7 Export cleaned data to a new csv file for future use

In [7]:
print("\n--- Exporting cleaned data ---")

# Create the processed data directory if it doesn't exist
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

# Export the cleaned data to a new CSV file, index=False to avoid writing row numbers
df.to_csv(PROCESSED_DATA_PATH, index=False, encoding='utf-8')

print(f"Completed! Cleaned dataset size: {df.shape[0]} rows, {df.shape[1]} columns.")
print(f"File saved at: {PROCESSED_DATA_PATH}")


--- Exporting cleaned data ---
Completed! Cleaned dataset size: 952 rows, 25 columns.
File saved at: ../datasets/processed/cleaned_spotify_2023.csv
